# 08 — Prediction on New Data
Load the best saved model and predict plant health from new sensor readings.

In [1]:
import pandas as pd
import numpy as np
import joblib
import warnings
warnings.filterwarnings('ignore')

# Load best model, scaler, and encoder
model   = joblib.load('../../models/hybrid_stacking_model.pkl')
scaler  = joblib.load('../../models/scaler.pkl')
le      = joblib.load('../../models/label_encoder.pkl')
print("Model, scaler, and encoder loaded.")

Model, scaler, and encoder loaded.


In [2]:
# Predict function
def predict_plant_health(sensor_readings: dict) -> str:
    feature_order = [
        'Soil_Moisture', 'Ambient_Temperature', 'Soil_Temperature',
        'Humidity', 'Light_Intensity', 'Soil_pH', 'Nitrogen_Level',
        'Phosphorus_Level', 'Potassium_Level', 'Chlorophyll_Content',
        'Electrochemical_Signal'
    ]
    input_array = np.array([[sensor_readings[f] for f in feature_order]])
    input_scaled = scaler.transform(input_array)
    prediction = model.predict(input_scaled)[0]
    label = le.inverse_transform([prediction])[0]
    proba = model.predict_proba(input_scaled)[0]
    confidence = round(max(proba) * 100, 2)
    return label, confidence

In [3]:
# Example 1 — Healthy plant
sample_healthy = {
    'Soil_Moisture': 60.0, 'Ambient_Temperature': 22.0,
    'Soil_Temperature': 20.0, 'Humidity': 45.0,
    'Light_Intensity': 600.0, 'Soil_pH': 6.5,
    'Nitrogen_Level': 55.0, 'Phosphorus_Level': 40.0,
    'Potassium_Level': 50.0, 'Chlorophyll_Content': 38.0,
    'Electrochemical_Signal': 0.3
}
label, conf = predict_plant_health(sample_healthy)
print(f"Prediction : {label}")
print(f"Confidence : {conf}%")

Prediction : Healthy
Confidence : 98.51%


In [4]:
# Example 2 — High Stress plant
sample_stress = {
    'Soil_Moisture': 15.0, 'Ambient_Temperature': 35.0,
    'Soil_Temperature': 33.0, 'Humidity': 88.0,
    'Light_Intensity': 200.0, 'Soil_pH': 4.8,
    'Nitrogen_Level': 8.0, 'Phosphorus_Level': 5.0,
    'Potassium_Level': 10.0, 'Chlorophyll_Content': 12.0,
    'Electrochemical_Signal': 0.95
}
label, conf = predict_plant_health(sample_stress)
print(f"Prediction : {label}")
print(f"Confidence : {conf}%")

Prediction : High Stress
Confidence : 99.43%


In [5]:
# Batch prediction example
batch = pd.DataFrame([sample_healthy, sample_stress])
feature_order = [
    'Soil_Moisture', 'Ambient_Temperature', 'Soil_Temperature',
    'Humidity', 'Light_Intensity', 'Soil_pH', 'Nitrogen_Level',
    'Phosphorus_Level', 'Potassium_Level', 'Chlorophyll_Content',
    'Electrochemical_Signal'
]
batch_scaled = scaler.transform(batch[feature_order])
batch_preds = le.inverse_transform(model.predict(batch_scaled))
batch['Predicted_Status'] = batch_preds
print(batch[['Soil_Moisture', 'Humidity', 'Chlorophyll_Content', 'Predicted_Status']])

   Soil_Moisture  Humidity  Chlorophyll_Content Predicted_Status
0           60.0      45.0                 38.0          Healthy
1           15.0      88.0                 12.0      High Stress
